# 08 Solution Exploration & Experiment Design
**Project:** E-Commerce Product Analytics ? Search & Conversion Funnel  
**Workflow:** Solution Exploration & Experiment Design  
**Focus:** Moving from Prioritized Problem (Search Discovery Failure) to Evaluated Solutions, MVP Selection, and Experiment Design.


In [ ]:
import sys, os, math
import duckdb
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import Image, display

DB_PATH = os.path.join('..', 'data', 'ecommerce_analytics.duckdb')
conn = duckdb.connect(DB_PATH, read_only=True)

def q(sql):
    return conn.execute(sql).df()

print("Connected to DuckDB successfully!")


## 1. Problem A Baseline Verification (Search Discovery Failure)
We verify the analytical facts discovered in exploratory analysis and prioritized:
- 4+ token queries experience an 8.23% Zero-Result Rate (ZRR) vs 1.78% for 1-3 token queries.
- CTR on 4+ token queries drops to 62.97% vs 70.77% for 1-3 token queries.


In [ ]:
df_search_baseline = q('''
WITH token_buckets AS (
    SELECT 
        CASE 
            WHEN ARRAY_LENGTH(STRING_SPLIT(TRIM(query_text), ' ')) >= 4 THEN 'Specific (4+ tokens)'
            ELSE 'Broad (1-3 tokens)'
        END AS query_type,
        COUNT(*) AS total_searches,
        SUM(is_zero_result) AS zero_results,
        ROUND(AVG(is_zero_result) * 100, 2) AS zrr_pct,
        SUM(has_pdp_click) AS clicked_searches,
        ROUND(AVG(has_pdp_click) * 100, 2) AS ctr_pct,
        ROUND(AVG(is_reformulation) * 100, 2) AS reform_pct
    FROM search_events
    GROUP BY 1
)
SELECT * FROM token_buckets ORDER BY total_searches DESC;
''')
display(df_search_baseline)


## 2. Session & User Reach for Specific Queries
Determine total eligible search sessions and users that formulate 4+ token queries.


In [ ]:
df_reach = q('''
WITH session_query_stats AS (
    SELECT 
        session_id,
        user_id,
        MAX(CASE WHEN ARRAY_LENGTH(STRING_SPLIT(TRIM(query_text), ' ')) >= 4 THEN 1 ELSE 0 END) AS has_specific_query
    FROM search_events
    GROUP BY 1, 2
)
SELECT 
    has_specific_query,
    COUNT(DISTINCT session_id) AS sessions,
    COUNT(DISTINCT user_id) AS users
FROM session_query_stats
GROUP BY 1;
''')
display(df_reach)


## 3. Solution Brainstorming & Scoring
We evaluated 10 candidate solution concepts using the formula:
$$\text{Solution Score} = \frac{\text{Impact} \times \text{Reach} \times \text{Confidence}}{\text{Effort}}$$


In [ ]:
df_sol = pd.read_csv('../reports/solution_prioritization.csv')
display(df_sol[['Rank', 'Solution_ID', 'Solution_Name', 'Category', 'Impact', 'Reach', 'Confidence', 'Effort', 'Solution_Score', 'Qualitative_Risk']])


## 4. Visualizing Solution Prioritization
Figure 16 displays the ranked solution scores alongside an Impact vs. Effort prioritization quadrant.


In [ ]:
display(Image('../reports/figures/16_solution_prioritization.png'))


## 5. MVP User Journey: Before vs. After
The selected MVP is **SOL-01: Query Relaxation (Soft-Match Fallback)**.
Figure 18 illustrates how the user journey transforms from rigid failure to successful product discovery.


In [ ]:
display(Image('../reports/figures/18_user_journey_before_after.png'))


## 6. A/B Experiment Sizing & Power Analysis
For **EXP-01: Automated Soft Query Relaxation**, we calculate the required sample size per variant for a two-tailed proportion test on Search-to-PDP CTR:
- Significance level $\alpha = 0.05$ ($Z_{\alpha/2} = 1.96$)
- Statistical power $1 - \beta = 0.80$ ($Z_\beta = 0.84$)
- Baseline CTR $p_1 = 0.6297$
- Tested MDEs: +2.0 pp, +3.0 pp, +4.0 pp, +5.0 pp


In [ ]:
def calculate_sample_size(p1, mde, alpha=0.05, power=0.80):
    p2 = p1 + mde
    p_bar = (p1 + p2) / 2.0
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    
    numerator = (z_alpha * math.sqrt(2 * p_bar * (1 - p_bar)) + z_beta * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    denominator = (p2 - p1) ** 2
    return math.ceil(numerator / denominator)

baseline_ctr = 0.6297
mde_list = [0.02, 0.03, 0.04, 0.05]
daily_eligible_queries = 10914 / 60  # Assuming 60 days in dataset

sizing_results = []
for mde in mde_list:
    n_per_variant = calculate_sample_size(baseline_ctr, mde)
    total_n = n_per_variant * 2
    est_days = math.ceil(total_n / daily_eligible_queries)
    sizing_results.append({
        'MDE (pp lift)': f'+{mde*100:.1f} pp',
        'Target CTR': f'{(baseline_ctr + mde)*100:.2f}%',
        'Sample Size / Variant': f'{n_per_variant:,}',
        'Total Sample Size': f'{total_n:,}',
        'Est. Days (at ~182 searches/day)': est_days
    })

df_sizing = pd.DataFrame(sizing_results)
display(df_sizing)


## 7. Success Metric Tree
Figure 17 maps high-level business goals to user outcomes, the primary experiment metric, supporting funnel metrics, and operational guardrails.


In [ ]:
display(Image('../reports/figures/17_experiment_metric_tree.png'))


## 8. Experiment Matrix
Complete suite of planned experiments ranked by priority and testability.


In [ ]:
df_exp = pd.read_csv('../reports/experiment_matrix.csv')
display(df_exp[['Rank', 'Experiment_ID', 'Experiment_Name', 'Primary_Metric', 'Risk', 'Effort']])
conn.close()
print("Solution exploration analysis complete!")
